In [ ]:
print('hi')

In [43]:
import pandas as pd
import tensorflow as tf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [44]:
df=pd.read_csv('insurance_data.csv')
df.head()

,age,affordibility,bought_insurance
0,22,1,0
1,25,0,0
2,47,1,1
3,52,0,0
4,46,1,1


In [45]:
x_train, x_test,y_train , y_test = train_test_split(df[['age','affordibility']], df['bought_insurance'], test_size=0.3, random_state=42)

In [ ]:
x_test

In [ ]:
x_train_scale = x_train.copy()
x_test_scale = x_test.copy()


x_train_scale['age'] = x_train_scale['age']/100

x_test_scale['age'] = x_test_scale['age']/100

In [34]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1,input_shape=(2,),activation='sigmoid',kernel_initializer='ones',bias_initializer='zero')
])

model.compile(
     optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

model.fit(x_train_scale ,y_train, epochs=5000)

model.evaluate(x_test_scale,y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - accuracy: 0.8947 - loss: 0.5477
Epoch 3150/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.8947 - loss: 0.5477
Epoch 3151/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.8947 - loss: 0.5476
Epoch 3152/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.8947 - loss: 0.5476
Epoch 3153/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.8947 - loss: 0.5476
Epoch 3154/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.8947 - loss: 0.5475
Epoch 3155/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.8947 - loss: 0.5475
Epoch 3156/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.8947 - loss: 0.5475
Epoch 3157/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - accuracy: 0.8947 - loss: 0.5474
Epoch 3158/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.8947 - loss: 0.5474
Epoch 3159/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.8947 - loss: 0.5474
Epoch 3160/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14

[0.33600708842277527, 1.0]

In [76]:
y_pred = model.predict(x_test_scale)
y_pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step


array([[0.80006576],
       [0.74104965],
       [0.80761725],
       [0.19024163],
       [0.38195688],
       [0.19772974],
       [0.7760886 ],
       [0.59408605],
       [0.45168307]], dtype=float32)

In [77]:
x_train_scale

,age,affordibility
13,0.29,0
15,0.55,1
1,0.25,0
4,0.46,1
5,0.56,1
2,0.47,1
16,0.25,0
23,0.45,1
3,0.52,0
26,0.23,1


In [38]:
y_test


9     1
25    1
8     1
21    0
0     0
12    0
17    1
22    1
11    0
Name: bought_insurance, dtype: int64

In [72]:
coef,intercept = model.get_weights()
coef,intercept

(array([[4.7896295],
        [1.1587756]], dtype=float32),
 array([-2.6937444], dtype=float32))

In [74]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

In [75]:
def pred_fun(age,aff):
    weight_sum = coef[0] * age + coef[1] * aff + intercept
    return   sigmoid(weight_sum)

In [79]:
pred_fun(0.52,0)

array([0.44938964], dtype=float32)

In [83]:
def bce_loss(y_true,y_pred):
    epsilon = 1e-15
    y_predicted_new = [max(i,epsilon) for i in y_pred]
    y_predicted_new = [min(i,1-epsilon) for i in y_predicted_new]

    y_predicted_new = np.array(y_predicted_new)

    return -np.mean(y_true*np.log(y_predicted_new)+(1-y_true)* np.log((1-y_predicted_new)))

In [84]:
def sigmoid_numpy(x):
    return 1/(1+np.exp(-x))
sigmoid_numpy(np.array([12,0,1]))

array([0.99999386, 0.5       , 0.73105858])

In [85]:
def gradient_desenct(age,affordibility,y_true,epochs,termater):
    w1=w2 = 1
    bias =0
    rate =0.5
    n= len(age)
    for i in range(epochs):
        weight_sum = w1*age + w2*affordibility+bias
        y_predicted = sigmoid(weight_sum)
        loss = bce_loss(y_true,y_predicted)
        w1d = (1/n) * np.dot(np.transpose(age),(y_predicted - y_true))
        w2d = (1/n) * np.dot(np.transpose(affordibility),(y_predicted - y_true))

        bias_d = np.mean(y_predicted - y_true)

        w1 = w1 - rate * w1d
        w2 = w2 - rate * w2d

        bias = bias - rate* bias_d

        print(f'epoch: {i}, weight1: {w1}, weght2: {w2}, bias: {bias},loss: {loss}')
        if loss <=termater:
            break
    return w1,w2,bias

In [86]:
gradient_desenct(x_train_scale['age'],x_train_scale['affordibility'],y_train,1000,0.5031)

epoch: 0, weight1: 0.9706119767414991, weght2: 0.931445943841482, bias: -0.12515001448472837,loss: 0.7583791301181848
epoch: 1, weight1: 0.9477665733269814, weght2: 0.8738300562014687, bias: -0.23327627964612652,loss: 0.7189537081387197
epoch: 2, weight1: 0.9309387507777787, weght2: 0.8265151276537606, bias: -0.3258025472147187,loss: 0.6902785819969751
epoch: 3, weight1: 0.9194599538404714, weght2: 0.7885081899949917, bias: -0.4044994989490006,loss: 0.6699077589779453
epoch: 4, weight1: 0.9126078775397289, weght2: 0.7586390667434824, bias: -0.4712554420328752,loss: 0.6556632975024537
epoch: 5, weight1: 0.9096749978419953, weght2: 0.7357062942653932, bias: -0.5279045197820315,loss: 0.6457773915946133
epoch: 6, weight1: 0.910010996028176, weght2: 0.7185737719152882, bias: -0.5761226465447706,loss: 0.6389086033792196
epoch: 7, weight1: 0.9130430759616713, weght2: 0.7062225253451166, bias: -0.617379578063776,loss: 0.6340872320101386
epoch: 8, weight1: 0.9182810900753401, weght2: 0.69777012

(4.788331211473521, 1.191193009284205, -2.728165195361072)

In [18]:
import tensorflow as tf
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10)
])
model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5)
model.evaluate(x_test, y_test)

c:\Users\EATSK\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9131 - loss: 0.2992
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9576 - loss: 0.1442
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9672 - loss: 0.1091
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9725 - loss: 0.0894
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9766 - loss: 0.0759
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9764 - loss: 0.0753


[0.07533177733421326, 0.9764000177383423]

In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305,312 (1.16 MB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 203,542 (795.09 KB)

In [2]:
print('hi')

hi


In [3]:
import tensorflow as tf
from tensorflow.keras import datasets

In [4]:
(x_train,y_train), (x_test,y_test) = datasets.cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 177s 1us/step


In [10]:
class DepthwiseSeparableConv2D(tf.keras.layers.Layer):
  def __init__(self, filters, kernel_size, padding, activation):
    super(DepthwiseSeparableConv2D, self).__init__()
    self.depthwise = tf.keras.layers.DepthwiseConv2D(kernel_size = kernel_size, padding = padding, activation = activation)
    self.pointwise = tf.keras.layers.Conv2D(filters = filters, kernel_size = (1, 1), activation = activation)

  def call(self, input_tensor):
    x = self.depthwise(input_tensor)
    return self.pointwise(x)

In [11]:
visible = tf.keras.Input(shape=(32, 32, 3))
depthwise_separable = DepthwiseSeparableConv2D(filters=64, kernel_size=(3,3), padding="valid", activation="relu")(visible)
depthwise_model = tf.keras.Model(inputs=visible, outputs=depthwise_separable)
depthwise_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_separable_conv2d      │ (None, 30, 30, 64)     │           286 │
│ (DepthwiseSeparableConv2D)      │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 286 (1.12 KB)

 Trainable params: 286 (1.12 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
normal = tf.keras.layers.Conv2D(filters=64,kernel_size=(3,3),padding='valid',activation='relu')(visible)
depthwise_model = tf.keras.Model(inputs=visible, outputs=normal)
depthwise_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 30, 30, 64)     │         1,792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,792 (7.00 KB)

 Trainable params: 1,792 (7.00 KB)

 Non-trainable params: 0 (0.00 B)

In [2]:
import tensorflow as tf
from tensorflow.keras import layers,models

In [25]:
model = models.Sequential([
    layers.Input(shape=(64,64,3)),
    layers.SeparableConv2D(
        filters =32,
        kernel_size=(3, 3),
        padding ='same',
        activation='relu'
    ),
    layers.SeparableConv2D(64,(3,3),padding='same',activation='relu'),

    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(10,activation='softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ separable_conv2d                │ (None, 64, 64, 32)     │           155 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_1              │ (None, 64, 64, 64)     │         2,400 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 65536)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │       655,370 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 657,925 (2.51 MB)

 Trainable params: 657,925 (2.51 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
def depthwise_separable_block(x,padding,kernal_size=(3,3)):
    

In [26]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
# Normalize pixel values to 0-1
x_train = x_train / 255.0
x_test = x_test / 255.0

In [27]:
model = models.Sequential([
    layers.Input(shape=(32,3,3)),
    layers.Conv2D(32,(3,3),padding='same',activation='relu'),

    layers.SeparableConv2D(64,(3,3),padding='same',activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.SeparableConv2D(128,(3,3),padding='same',activation='relu'),
    layers.SeparableConv2D(128,(3,3),padding='same',activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.SeparableConv2D(256,(3,3),padding='same',activation='relu'),
    layers.GlobalAvgPool2D(),

    layers.Dense(256,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10,activation='softmax')
])

model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(x_train,y_train,epochs=10,batch_size=64,validation_data=(x_test,y_test))

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 57s 69ms/step - accuracy: 0.0970 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 54s 69ms/step - accuracy: 0.0972 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 53s 68ms/step - accuracy: 0.0998 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 85s 71ms/step - accuracy: 0.0986 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 54s 69ms/step - accuracy: 0.0981 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 52s 66ms/step - accuracy: 0.0995 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 53s 67ms/step - accuracy: 0.0955 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 52s 67ms/step - accuracy: 0.0980 - loss: 2.3027 - 

In [28]:
model.evaluate(x_test,y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.1000 - loss: 2.3026


[2.3026084899902344, 0.10000000149011612]

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Load CIFAR-10 dataset (tiny images of cats, dogs, planes, etc.)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values to 0-1
x_train = x_train / 255.0
x_test = x_test / 255.0

# Build a MobileNet-style model (uses depthwise separable convs)
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    
    # First regular conv
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    
    # Depthwise Separable blocks
    layers.SeparableConv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.SeparableConv2D(128, (3, 3), padding='same', activation='relu'),
    layers.SeparableConv2D(128, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.SeparableConv2D(256, (3, 3), padding='same', activation='relu'),
    layers.GlobalAveragePooling2D(),
    
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train!
model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(x_test, y_test)
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 58s 69ms/step - accuracy: 0.0983 - loss: 2.3029 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 2/10


In [3]:
import tensorflow as tf
from tensorflow.keras import layers,models,datasets

In [4]:
(x_train,y_train),(x_test,y_test) = datasets.mnist.load_data()

In [10]:
model = tf.keras.Sequential([
    layers.Input(shape=(32,32,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.SeparableConv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.SeparableConv2D(128,(3,3),activation='relu'),
    layers.SeparableConv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Dense(128,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10,activation='sigmoid')
])

model.compile(
    opitmizer = 'adam',
    loss = "sparse_categorical_crossentropy",
    metrics = ['accuracy']
)


TypeError: Trainer.compile() got an unexpected keyword argument 'opitmizer'